# Week 9: Transformers & LLM Architectures
## Notebook 11: Mistral 7B Sliding Window Attention (SWA) & Ring Buffers

### 1. Conceptual Intuition & Theoretical Background
In standard Multi-Head Attention, memory and compute scale quadratically $O(S^2)$ with sequence length $S$.

**Sliding Window Attention (SWA, Beltagy et al., 2020; Mistral, 2023):**
Restricts each token to attend to at most $W$ preceding tokens:
$$\text{Attention Matrix Mask } M_{i,j} = \begin{cases} 0 & \text{if } 0 \le i - j \le W \\ -\infty & \text{otherwise} \end{cases}$$

**Theoretical Receptive Field:**
Although an individual attention layer only sees a local window of $W$ tokens, information propagates hierarchically across layers:
$$\text{Effective Receptive Field} = L \times W$$
For a model with $L = 32$ layers and window size $W = 4,096$, the effective receptive field reaches $131,072$ tokens!

**Rolling Buffer KV-Cache:**
Caches are stored in a fixed circular ring buffer of size $W$:
$$\text{Buffer Index} = i \pmod W$$
Capping inference VRAM usage at $W$ tokens regardless of total generated length.

---

### 2. Scratch Python Implementation: SWA Ring Buffer Engine

In [ ]:
import numpy as np

class SlidingWindowAttentionScratch:
    def __init__(self, window_size=4, head_dim=16):
        self.W = window_size
        self.head_dim = head_dim
        
    def forward(self, Q, K, V):
        """
        Q, K, V: shape (seq_len, head_dim)
        """
        S, d = Q.shape
        # Compute dot-product attention
        scores = np.dot(Q, K.T) / np.sqrt(d)
        
        # Build Sliding Window Attention Mask
        indices = np.arange(S)
        diff = indices[:, np.newaxis] - indices[np.newaxis, :]
        
        # Token i can only attend to j if: 0 <= i - j < W
        swa_mask = (diff >= 0) & (diff < self.W)
        
        # Apply mask: set invalid tokens to -infinity
        masked_scores = np.where(swa_mask, scores, -1e9)
        
        # Softmax
        exp_scores = np.exp(masked_scores - np.max(masked_scores, axis=-1, keepdims=True))
        weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
        
        out = np.dot(weights, V)
        return out, swa_mask

# Simulate Sequence of 10 Tokens with Window W = 3
seq_len = 10
window_size = 3
head_dim = 8

np.random.seed(42)
Q = np.random.randn(seq_len, head_dim)
K = np.random.randn(seq_len, head_dim)
V = np.random.randn(seq_len, head_dim)

swa = SlidingWindowAttentionScratch(window_size=window_size, head_dim=head_dim)
out, mask = swa.forward(Q, K, V)

print("--- Sliding Window Attention (W=3) ---")
print(f"Sequence Length: {seq_len} | Sliding Window: {window_size}")
print("\nSWA Attention Connectivity Mask (1=Attends, 0=Masked):")
print(mask.astype(int))
print(f"\nToken 9 Attends Strictly to Tokens: {np.where(mask[9])[0].tolist()}")
print("KV-Cache memory capped at W elements using rolling modulo ring buffer!")

### 3. Step-by-Step Code Walkthrough
1. **Sliding Mask:** Limits non-zero attention weights to the band $0 \le i - j < W$.
2. **Constant Cache Size:** Tokens older than $W$ are overwritten via circular ring buffer indexing ($t \pmod W$).
3. **Hierarchical Stacking:** Receptive field expands linearly ($L \times W$) as layer depth increases.

---

### 4. Industry Applications & Trade-offs
- **Mistral-7B & Mixtral-8x7B:** Enabled handling 32K context windows with minimal VRAM footprints.
- **Trade-off:** Attention cannot directly retrieve isolated needle tokens beyond window $W$ in a single step.

---

### 5. Top Interview Questions & Answers
**Q1: How does Sliding Window Attention achieve a global receptive field of $131K$ tokens with a local window of $4K$?**
> **Answer:** Information flows hierarchically: layer 1 gathers context from tokens $t-W$ to $t$. Layer 2 gathers information from layer 1, extending its effective context to $t-2W$. Across $L$ layers, the theoretical receptive field equals $L \times W$.

**Q2: How does the Rolling Buffer Cache in Mistral save GPU memory during autoregressive generation?**
> **Answer:** Keys and values are written to a fixed buffer of size $W$ at position $i \pmod W$. Once sequence length exceeds $W$, new tokens overwrite the oldest slots, keeping memory consumption constant.